# 완전한 비즈니스 솔루션

### 비즈니스 과제:

예비 고객, 투자자, 그리고 잠재적 채용 대상자에게 사용할 회사 소개 브로슈어를 만들어주는 제품을 만드세요.

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

In [16]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-3.1-flash-lite' 

gemini = OpenAI(
    base_url = GEMINI_BASE_URL,
    api_key = api_key
)

In [18]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog/zh',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 'https://blogs.nvidia.com/blog/nvidia-to-acquire-hugging-face/',
 '/spaces',
 '/models',
 '/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp',
 '/Qwen/Qwen3.8-27B',
 '/XHToken/Spark-X2.5-4B',
 '/Qwen/Qwen3.8-Flash-Next',
 '/google/timesfm-3.0-pytorch',
 '/models',
 '/spaces/pollen-robotics/microduck-simulator',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/kulkas2pintu/QWEN_EDIT_IMAGE',
 '/spaces/MrdDickDickenson/Krea-2-Turbo_I2I',
 '/spaces/AimeeBingmouQu/ProtectBirds',
 '/spaces',
 '/datasets/rajpurkar/squad',
 '/datasets/stanfordnlp/imdb',
 '/datasets/nyu-mll/glue'

In [11]:
link_system_prompt = """
당신에게는 어떤 웹페이지에서 찾은 링크 목록이 주어집니다.
당신은 이 링크들 중 어떤 것이 회사 소개 브로슈어에 포함하기에 가장 적합한지
판단할 수 있습니다. 예를 들어 회사 소개(About) 페이지, 회사(Company) 페이지,
채용(Careers/Jobs) 페이지로 연결되는 링크 등이 있습니다.
다음 예시와 같이 JSON 형식으로 응답해야 합니다:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
다음은 웹사이트 {url}에 있는 링크 목록입니다 -
이 중에서 회사 소개 브로슈어에 적합한 웹 링크를 판단하여,
전체 https URL을 JSON 형식으로 응답해 주세요.
이용약관(Terms of Service), 개인정보처리방침(Privacy), 이메일 링크는 포함하지 마세요.

링크 목록 (일부는 상대 경로 링크일 수 있습니다):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [19]:
print(get_links_user_prompt("https://huggingface.co"))


다음은 웹사이트 https://huggingface.co에 있는 링크 목록입니다 -
이 중에서 회사 소개 브로슈어에 적합한 웹 링크를 판단하여,
전체 https URL을 JSON 형식으로 응답해 주세요.
이용약관(Terms of Service), 개인정보처리방침(Privacy), 이메일 링크는 포함하지 마세요.

링크 목록 (일부는 상대 경로 링크일 수 있습니다):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog/zh
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
https://blogs.nvidia.com/blog/nvidia-to-acquire-hugging-face/
/spaces
/models
/deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
/Qwen/Qwen3.8-27B
/XHToken/Spark-X2.5-4B
/Qwen/Qwen3.8-Flash-Next
/google/timesfm-3.0-pytorch
/models
/spaces/pollen-robotics/microduck-simulator
/spaces/kulkas2pintu/wan555
/spaces/kulkas2pintu/QWEN_EDIT_IMAGE
/spaces/MrdDickDickenson/Krea-2-Turbo_I2I
/spaces/AimeeBingmouQu/ProtectBirds
/spaces
/datasets/rajpurkar/squad
/datasets/stanfordnlp/imdb
/

In [20]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [21]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'}]}

# 다음 스텝

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
×
We are happy to share our intention to join forces with
NVIDIA
.
Read the announcement
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
Updated
5 days ago
•
209k
•
698
Qwen/Qwen3.8-27B
Updated
23 days ago
•
6.19M
•
14.1k
XHToken/Spark-X2.5-4B
Updated
3 days ago
•
5.48k
•
567
Qwen/Qwen3.8-Flash-Next
Updated
10 days ago
•
433k
•
4.93k
google/timesfm-3.0-pytorch
Updated
4 days ago
•
144k
•

In [26]:
brochure_system_prompt = """
당신은 회사 웹사이트에서 가져온 여러 관련 페이지의 내용을 분석하여
예비 고객, 투자자, 채용 대상자를 위한 짧은 회사 소개 브로슈어를 작성하는 어시스턴트입니다.
코드 블록 없이 마크다운 형식으로 응답하세요.
가지고 있는 정보가 있다면 회사 문화, 고객, 채용/직무에 대한 내용을 포함하세요.
"""

# 좀 더 유머러스한 브로슈어를 원한다면 아래 줄의 주석을 해제하세요 - 이는 '톤'을 얼마나 쉽게 적용할 수 있는지 보여주는 예시입니다:

# brochure_system_prompt = """
# 당신은 회사 웹사이트에서 가져온 여러 관련 페이지의 내용을 분석하여
# 예비 고객, 투자자, 채용 대상자를 위한 짧고 유머러스하며 재치 있는 회사 소개 브로슈어를 작성하는 어시스턴트입니다.
# 코드 블록 없이 마크다운 형식으로 응답하세요.
# 가지고 있는 정보가 있다면 회사 문화, 고객, 채용/직무에 대한 내용을 포함하세요.
# """

In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
당신은 {company_name}이라는 회사를 살펴보고 있습니다.
다음은 이 회사의 랜딩 페이지와 기타 관련 페이지들의 내용입니다;
이 정보를 바탕으로 코드 블록 없이 마크다운 형식으로 이 회사에 대한 짧은 브로슈어를 작성하세요.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # 5,000자를 초과하면 잘라냄
    return user_prompt

In [25]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\n당신은 HuggingFace이라는 회사를 살펴보고 있습니다.\n다음은 이 회사의 랜딩 페이지와 기타 관련 페이지들의 내용입니다;\n이 정보를 바탕으로 코드 블록 없이 마크다운 형식으로 이 회사에 대한 짧은 브로슈어를 작성하세요.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\n×\nWe are happy to share our intention to join forces with\nNVIDIA\n.\nRead the announcement\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Flash-Vision-Exp\nUpdated\n5 days ago\n•\n209k\n•\n698\nQwen/Qwen3.8-27B\nUpdated\n2

In [27]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [28]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: 인공지능 미래를 함께 만드는 협업의 허브

## 회사 소개
Hugging Face는 머신러닝 커뮤니티가 인공지능의 미래를 함께 설계하고 구축하는 중심 플랫폼입니다. 전 세계 개발자와 연구자들이 오픈 소스 모델, 데이터셋, 그리고 AI 애플리케이션을 공유하고 실험하며 협업할 수 있는 기술 생태계의 핵심적인 역할을 수행하고 있습니다. 200만 개 이상의 모델과 50만 개 이상의 데이터셋이 호스팅되어 있는 Hugging Face는 '머신러닝의 집'으로서 그 영향력을 확장하고 있습니다.

## 주요 비즈니스 및 솔루션
Hugging Face는 개인 연구자부터 대규모 기업까지 아우르는 다양한 플랫폼 도구를 제공합니다:

*   **모델 및 데이터 허브:** 오픈 소스 머신러닝 모델과 방대한 데이터셋을 탐색, 활용, 공유할 수 있는 통합 공간입니다.
*   **Spaces 및 애플리케이션:** 구축된 AI 모델을 시각화하고 실제 애플리케이션으로 바로 구현할 수 있는 환경을 제공합니다.
*   **엔터프라이즈 지원:** 기업을 위한 협업 도구, 전용 인프라, 추론 엔드포인트(Inference Endpoints) 및 스토리지 솔루션을 통해 비즈니스 규모에 맞춘 AI 배포를 지원합니다.
*   **협업 인프라:** 머신러닝 프로젝트의 관리와 배포를 가속화하는 다양한 통합 서비스(Hugging Face PRO, 엔터프라이즈 서포트 등)를 운영합니다.

## 회사 문화 및 가치
Hugging Face는 '개방성'과 '커뮤니티'를 최우선 가치로 삼습니다. 기술의 장벽을 낮추고, 누구나 쉽게 최첨단 AI 기술에 접근할 수 있도록 돕는 것이 우리의 사명입니다. 
*   **공유와 협업:** 전 세계 커뮤니티와의 긴밀한 소통(Discord, 포럼 등)을 통해 기술 발전을 앞당깁니다.
*   **오픈 소스 정신:** 지식과 자원을 투명하게 공유함으로써 머신러닝의 표준을 세우고 생태계의 성장을 견인합니다.
*   **지속적 혁신:** 기술의 중심에서 NVIDIA와 같은 글로벌 선도 기업들과 협력하며, 더 나은 AI 미래를 구축하기 위한 도전을 멈추지 않습니다.

## 인재 채용 및 커리어
우리는 인공지능이라는 거대한 변화를 함께 만들어갈 열정적인 인재를 기다립니다. Hugging Face에서 일한다는 것은 단순히 코드를 작성하는 것을 넘어, 인공지능이 세상을 바꾸는 최전선에서 전 세계 수백만 명의 사용자가 사용하는 인프라를 설계하는 것을 의미합니다.

*   **우리가 찾는 동료:** AI와 머신러닝 생태계에 열정이 있으며, 개방적이고 협력적인 커뮤니티 문화에서 성장하고 싶은 분.
*   **함께하는 이유:** 
    *   글로벌 머신러닝 생태계의 표준을 만드는 현장에서 직접 기여할 수 있습니다.
    *   최첨단 기술 프로젝트를 통해 전문성을 극대화할 수 있습니다.
    *   자유롭고 역동적인 커뮤니티 중심의 업무 환경에서 성장합니다.

---
**더 나은 인공지능의 미래, Hugging Face와 함께 만들어가세요.**
웹사이트: [huggingface.co](https://huggingface.co)

In [29]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")